# Fig. S5: participation entropy and entanglement of eigenstates

S2, Q and S_half of the eigenstates of the largest subspace of the example
sectors of Figs. 3 and S3 (ferromagnetic network, ±J network, ring),
against energy and against each other (spectra.analyse_eigenstates).

Data: the Fig. 3 and Fig. S3 example files.

In [ ]:
%matplotlib inline


import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from qising_networks import FIGURES_DIR as FIGURES, PROJECT_ROOT
from qising_networks import fig_maker as fm
from qising_networks import scar_metrics as sm
from qising_networks import spectra
from qising_networks.model import GAMMA
from qising_networks.plots.dynamics import example_sectors

mpl.rc_file(PROJECT_ROOT / "matplotlibrc")

In [ ]:
systems = example_sectors()
results = [(name, spectra.analyse_eigenstates(*args, ring=(k == 2)))
           for k, (name, args) in enumerate(systems)]
for name, d in results:
    print(f"{name}: dim={d['dim']} n_atyp={d['atyp'].sum()} "
          f"r={d['r']:.3f} r_partial={d['r_partial']:.3f} "
          f"E0={d['E0']:.3f}")
    print(f"    S_half: cut={d['half_cut']} median={np.median(d['S_half']):.3f} "
          f"r(S_half, Q)={d['r_h']:.3f} r_partial={d['r_h_partial']:.3f}")
    for a in np.where(d['ent_outlier'])[0]:
        print(f"    S_half outlier: (E-E0)/Gamma={(d['E'][a] - d['E0']) / GAMMA:+.3f} "
              f"S_half={d['S_half'][a]:.3f} zS={d['zS'][a]:+.2f} "
              f"Q={d['Q'][a]:.3f} zQ={d['zQ'][a]:+.2f} S2-atypical={d['atyp'][a]}")


In [ ]:
C_ALL = '0.65'
C_SCAR = 'C3'
MS = 2.0


def scatter(ax, x, y, nt):
    ax.plot(x[~nt], y[~nt], 'o', ms=MS, mec='none', color=C_ALL,
            label='Eigenstates', rasterized=True)
    ax.plot(x[nt], y[nt], 'o', ms=MS, mec='none', color=C_SCAR,
            label=r'Atypical in $S_2$ (scar candidates)')


with plt.rc_context({'font.size': 7, 'legend.fontsize': 5}):
    fig = plt.figure(figsize=(fm.MAX_FIG_WIDTH, fm.MAX_FIG_HEIGHT),
                     constrained_layout=True)
    # Columns: networks; rows: S2, Q, S_half vs E, then S2 and S_half vs Q.
    grid = fig.subplots(5, 3)
    for (ax_s, ax_q, ax_e, ax_c, ax_v), (name, d) in zip(grid.T, results):
        ax_s.set_title(f"{name}\n(subspace dimension $M = {d['dim']}$)")
        E, s2, Q, nt = d['E'], d['s2'], d['Q'], d['atyp']

        scatter(ax_q, E, Q, nt)
        Eg = np.linspace(E.min(), E.max(), 400)
        ax_q.plot(Eg, 1 - ((Eg - d['E0']) / (d['N'] * GAMMA))**2, '--',
                  color='k', lw=0.7,
                  label=r'$1-[(E-E_0)/(N\mathit{\Gamma})]^2$')
        ax_q.set_xlabel(r'Eigenenergy $E/J$')
        ax_q.set_ylabel('Multipartite\nentanglement $Q$')
        ax_q.set_ylim(top=1.08)

        scatter(ax_s, E, s2, nt)
        # Upper bound ln M and the GOE value ln M - ln 3.
        ax_s.axhline(np.log(d['dim']), color='k', ls='--', lw=0.7,
                     label=r'$\ln(M)$ (upper bound)')
        ax_s.axhline(np.log(d['dim']) - np.log(3), color='k', ls=':',
                     lw=0.7, label=r'$\ln(M) - \ln(3)$ (GOE)')
        ax_s.axhline(d['cut'], color='k', ls='-.', lw=0.7,
                     label=r'Mean $-\,3\,$std')
        ax_s.set_xlabel(r'Eigenenergy $E/J$')
        ax_s.set_ylabel('Participation\nentropy $S_2$')

        ol = d['ent_outlier']

        scatter(ax_e, E, d['S_half'], nt)
        ax_e.plot(E[ol], d['S_half'][ol], 'o', ms=MS + 2, mfc='none',
                  mec='k', mew=0.6)
        ax_e.set_xlabel(r'Eigenenergy $E/J$')
        ax_e.set_ylabel('Entanglement\nentropy $S_\\mathrm{half}$')

        scatter(ax_c, Q, s2, nt)
        ax_c.text(0.04, 0.96,
                  f"$r = {d['r']:.2f}$\n"
                  f"$r_{{|E}} = {d['r_partial']:.2f}$",
                  transform=ax_c.transAxes, va='top', ha='left')
        ax_c.set_ylabel('Participation\nentropy $S_2$')
        ax_c.tick_params(labelbottom=False)

        ax_v.sharex(ax_c)
        scatter(ax_v, Q, d['S_half'], nt)
        ax_v.plot(Q[ol], d['S_half'][ol], 'o', ms=MS + 2, mfc='none',
                  mec='k', mew=0.6,
                  label=r'Atypical in $S_\mathrm{half}$ at fixed $E$')
        ax_v.text(0.04, 0.96,
                  f"$r = {d['r_h']:.2f}$\n"
                  f"$r_{{|E}} = {d['r_h_partial']:.2f}$",
                  transform=ax_v.transAxes, va='top', ha='left')
        ax_v.set_xlabel(r'Multipartite entanglement $Q$')
        ax_v.set_ylabel('Entanglement\nentropy $S_\\mathrm{half}$')
    axes = list(grid.ravel())  # row-major: a-c / d-f / g-i / j-l / m-o

    axes[3].legend(handles=axes[3].get_lines()[2:], frameon=False,
                   loc='upper left', borderaxespad=0.1)
    # Reference-line legend, with headroom in panel a.
    # Share y along each row; only the left column keeps y labels.
    for k in (1, 2):
        for row in range(5):
            axes[3 * row + k].sharey(axes[3 * row])
            axes[3 * row + k].set_ylabel('')
            axes[3 * row + k].tick_params(labelleft=False)
    axes[3].set_ylim(-0.05, 1.25)
    axes[6].set_ylim(-0.1, 4.0)
    axes[9].set_ylim(-0.2, 9.5)  # headroom for the r text
    axes[12].set_ylim(-0.1, 5.0)
    axes[6].legend(handles=axes[12].get_lines()[2:], frameon=False,
                   loc='upper left', borderaxespad=0.2)
    axes[0].set_ylim(-0.2, np.log(results[0][1]['dim']) + 5.0)
    for ax in (axes[2], axes[5], axes[8]):  # avoid clipped last E tick
        ax.set_xticks([11.5, 12.0, 12.5])
    axes[0].add_artist(axes[0].legend(
        handles=axes[0].get_lines()[2:], frameon=False,
        loc='upper left', borderaxespad=0.1))
    # Marker legend applies to every panel, so it sits in a.
    axes[0].legend(handles=axes[0].get_lines()[:2], frameon=False,
                   loc='upper right', handletextpad=0.3,
                   borderaxespad=0.2, markerscale=1.5)

    fm.align_panel_labels_to_yaxis(
        fig, axes, fm.journal_labels("abcdefghijklmno"), fontsize=8, bold=True)
    # Align the left-column labels.
    first = axes[::3]
    x_left = min(ax.transAxes.transform(ax.texts[-1].get_position())[0]
                 for ax in first)
    for ax in first:
        ax.texts[-1].set_x(ax.transAxes.inverted().transform((x_left, 0))[0])


In [ ]:
paths = fm.save_unlabeled(fig, FIGURES / 'fig-s05', 'fig-s05')
print(paths)


## Numbers quoted in the supplement

In [ ]:

for name, d in results:
    print(f"{name}: M = {d['dim']}, atypical in S2: {int(d['atyp'].sum())}, "
          f"median S2 {np.median(d['s2']):.2f} vs ln(M) - ln(3) = {np.log(d['dim']) - np.log(3):.2f}, "
          f"r(S_half, S2) {np.corrcoef(d['S_half'], d['s2'])[0, 1]:.2f}, "
          f"r(S_half, Q) {np.corrcoef(d['S_half'], d['Q'])[0, 1]:.2f}, "
          f"r_|E(S_half, Q) {d['r_h_partial']:.2f}, "
          f"atypical in S_half: {int(d['ent_outlier'].sum())}")

    print(f"    retained nondegenerate eigenstates: {d['n_eig']} of {d['dim']}")


In [ ]:
# S_half of the half-filled Dicke state (Q = 1) and the mean magnetization
# of the ring's Neel sector.
from math import comb, log
N_D = 12
p = np.array([comb(N_D // 2, k) * comb(N_D // 2, N_D // 2 - k)
              for k in range(N_D // 2 + 1)], float) / comb(N_D, N_D // 2)
print(f"Dicke(12, 6): S_half = {-(p * np.log(p)).sum():.3f}, "
      f"(N/2) ln 2 = {N_D // 2 * log(2):.3f}")
ring_sec = systems[2][1][2]
ring_states = sm.states_from_indices(ring_sec, 12)
print(f"ring sector (M = {len(ring_sec)}): mean magnetization "
      f"{ring_states.mean():.3f}, Q bound "
      f"{1 - (ring_states.mean(axis=0) ** 2).mean():.3f}")
# z-scores of the ring's scar tower outside the zero-energy manifold
ring = results[2][1]
x = (ring['E'] - ring['E0']) / GAMMA
tower = [int(np.argmin(np.abs(x - e))) for e in (-3.95, -2.67, -1.34, 1.34,
                                                 2.67, 3.95)]
print("ring tower (E-E0)/Gamma:", np.round(x[tower], 2),
      "z(S_half):", np.round(ring['zS'][tower], 2),
      "z(Q):", np.round(ring['zQ'][tower], 2))
